# 前向传播、反向传播和计算图

## 练习4.7.1

假设一些标量函数$f$的输入$\mathbf{X}$是$n \times m$矩阵。$f$相对于$\mathbf{X}$的梯度维数是多少？

**解答：** $n \times m$ 维。

## 练习4.7.2

向本节中描述的模型的隐藏层添加偏置项（不需要在正则化项中包含偏置项）。

1. 画出相应的计算图。
2. 推导正向和反向传播方程。

**解答：**

1.（计算图可参考本章正文。）
2. 正向传播按计算图的顺序逐项计算：

$$\hat{\mathbf{z}}=\mathbf{W}^{(1)} \mathbf{x}$$

（第一层线性变换）

$$\mathbf{z}=\mathbf{b}^{(1)}+ \hat{\mathbf{z}}$$

（加上偏置）

$$\mathbf{h}=\mathbf{\phi} (\mathbf{z})$$

（激活函数）

$$\hat{\mathbf{o}}=\mathbf{W}^{(2)} \mathbf{h}$$

（第二层线性变换）

$$\mathbf{o}=\mathbf{b}^{(2)}+ \hat{\mathbf{o}}$$

（加上偏置）

$$L=l(\mathbf{o}, y)$$

（计算损失）

$$s=\frac{\lambda}{2}\left(\left\|\mathbf{W}^{(1)}\right\|_F^2+\left\|\mathbf{W}^{(2)}\right\|_F^2\right)$$

（正则化项）

$$J=L+s$$

（目标函数）

反向传播的目的是计算梯度 $\frac{\partial J}{\partial \mathbf{W}^{(1)}}$、$\frac{\partial J}{\partial \mathbf{W}^{(2)}}$、$\frac{\partial J}{\partial \mathbf{b}^{(1)}}$、$\frac{\partial J}{\partial \mathbf{b}^{(2)}}$。

前两个式子和本章正文中给出的相同：

$$\frac{\partial J}{\partial \mathbf{W}^{(1)}} = \text{prod}\left(\frac{\partial J}{\partial \mathbf{z}}, \frac{\partial \mathbf{z}}{\partial \mathbf{W}^{(1)}}\right) + \text{prod}\left(\frac{\partial J}{\partial s}, \frac{\partial s}{\partial \mathbf{W}^{(1)}}\right) = \frac{\partial J}{\partial \mathbf{z}} \mathbf{x}^\top + \lambda \mathbf{W}^{(1)}.$$

$$\frac{\partial J}{\partial \mathbf{W}^{(2)}} = \text{prod}\left(\frac{\partial J}{\partial \mathbf{o}}, \frac{\partial \mathbf{o}}{\partial \mathbf{W}^{(2)}}\right) + \text{prod}\left(\frac{\partial J}{\partial s}, \frac{\partial s}{\partial \mathbf{W}^{(2)}}\right) = \frac{\partial J}{\partial \mathbf{o}} \mathbf{h}^\top + \lambda \mathbf{W}^{(2)}.$$

详细的中间推导如下：

第一步是计算目标函数 $J=L+s$ 相对于损失项 $L$ 和正则项 $s$ 的梯度：
$$\frac{\partial J}{\partial L} = 1, \quad \frac{\partial J}{\partial s} = 1.$$

接下来，根据链式法则计算目标函数关于输出层变量 $\mathbf{o}$ 的梯度：
$$\frac{\partial J}{\partial \mathbf{o}} = \frac{\partial J}{\partial L} \cdot \frac{\partial L}{\partial \mathbf{o}} = \frac{\partial L}{\partial \mathbf{o}} \in \mathbb{R}^q.$$

计算正则化项相对于两个参数的梯度：
$$\frac{\partial s}{\partial \mathbf{W}^{(1)}} = \lambda \mathbf{W}^{(1)}, \quad \frac{\partial s}{\partial \mathbf{W}^{(2)}} = \lambda \mathbf{W}^{(2)}.$$

现在可以计算最接近输出层的模型参数的梯度 $\frac{\partial J}{\partial \mathbf{W}^{(2)}} \in \mathbb{R}^{q \times h}$。使用链式法则得出：
$$\frac{\partial J}{\partial \mathbf{W}^{(2)}} = \frac{\partial J}{\partial \mathbf{o}} \cdot \frac{\partial \mathbf{o}}{\partial \mathbf{W}^{(2)}} + \frac{\partial J}{\partial s} \cdot \frac{\partial s}{\partial \mathbf{W}^{(2)}} = \frac{\partial J}{\partial \mathbf{o}} \mathbf{h}^\top + \lambda \mathbf{W}^{(2)}.$$

为了获得关于 $\mathbf{W}^{(1)}$ 的梯度，需要继续沿着输出层到隐藏层反向传播。关于隐藏层输出的梯度 $\frac{\partial J}{\partial \mathbf{h}} \in \mathbb{R}^h$ 由下式给出：
$$\frac{\partial J}{\partial \mathbf{h}} = \frac{\partial J}{\partial \mathbf{o}} \cdot \frac{\partial \mathbf{o}}{\partial \mathbf{h}} = {\mathbf{W}^{(2)}}^\top \frac{\partial J}{\partial \mathbf{o}}.$$

由于激活函数 $\phi$ 是按元素计算的，计算中间变量 $\mathbf{z}$ 的梯度 $\frac{\partial J}{\partial \mathbf{z}} \in \mathbb{R}^h$ 需要使用按元素乘法运算符，用 $\odot$ 表示：
$$\frac{\partial J}{\partial \mathbf{z}} = \frac{\partial J}{\partial \mathbf{h}} \cdot \frac{\partial \mathbf{h}}{\partial \mathbf{z}} = \frac{\partial J}{\partial \mathbf{h}} \odot \phi'(\mathbf{z}).$$

最后，可以得到最接近输入层的模型参数的梯度 $\frac{\partial J}{\partial \mathbf{W}^{(1)}} \in \mathbb{R}^{h \times d}$。根据链式法则：
$$\frac{\partial J}{\partial \mathbf{W}^{(1)}} = \frac{\partial J}{\partial \mathbf{z}} \cdot \frac{\partial \mathbf{z}}{\partial \mathbf{W}^{(1)}} + \frac{\partial J}{\partial s} \cdot \frac{\partial s}{\partial \mathbf{W}^{(1)}} = \frac{\partial J}{\partial \mathbf{z}} \mathbf{x}^\top + \lambda \mathbf{W}^{(1)}.$$

根据链式法则，偏置项的梯度为：

$$\frac{\partial J}{\partial \mathbf{b}^{(1)}} = \frac{\partial J}{\partial \mathbf{z}} \cdot \frac{\partial \mathbf{z}}{\partial \mathbf{b}^{(1)}} = \frac{\partial J}{\partial \mathbf{z}}.$$

$$\frac{\partial J}{\partial \mathbf{b}^{(2)}} = \frac{\partial J}{\partial \mathbf{o}} \cdot \frac{\partial \mathbf{o}}{\partial \mathbf{b}^{(2)}} = \frac{\partial J}{\partial \mathbf{o}}.$$

## 练习4.7.3

计算本节所描述的模型，用于训练和预测的内存空间。

**解答：** 训练需要存储的参数：$x,z,h,o,y,W^{(1)},W^{(2)}, \frac{\partial J}{\partial \mathbf{W}^{(1)}},\frac{\partial J}{\partial \mathbf{W}^{(2)}}$。

假设输入数据为n维$W^{(1)}$和$\displaystyle\frac{\partial J}{\partial \mathbf{W}^{(1)}}$为$n \times m$维,则$z$和$h$为$m$维。$W^{(2)}$和$\displaystyle\frac{\partial J}{\partial \mathbf{W}^{(2)}}$为$m \times k$维,则$o$和$y$为$k$维。网络参数为浮点小数，通常用float单精度表示，单精度float占32位/4个字节。那么占用总字节数为$(n+n \times m \times 2+2 \times m+m \times k \times 2+k \times 2) \times 4B$

预测需要存储的参数：$x,z,h,o,y,W^{(1)},W^{(2)}$可以估计出占用内存总字节数为$(n+n \times m+2 \times m+m \times k+k \times 2) \times 4B$

## 练习4.7.4

假设想计算二阶导数。计算图发生了什么？预计计算需要多长时间？

**解答：** 需要再构造一个以一阶导数为正向传播的计算图，然后再反向传播求导。可能会花费相对与计算一阶导数时两倍的时间。

## 练习4.7.5

假设计算图对当前拥有的GPU来说太大了。
1. 请试着把它划分到多个GPU上。
2. 与小批量训练相比，有哪些优点和缺点？

**解答：**
1. 我们可以把网络按三种方式把它划分到GPU上,网络并行、分层并行、数据并行。网络并行把每层网络的计算划分到不同的GPU。分层并行把每层内的计算划分到不同的GPU,比如把全连接层输出单元拆分到不同gpu上计算。数据并行把数据拆分分别在不同gpu上计算同样的网络然后汇总各个gpu上更新参数。
2. 优点是多个GPU集群可以训练较大的模型（前两种划分方法可以）以及更快的训练模型（第三种划分），但是缺点是可能会因为节点间通信的限制导致速度不够快。

---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
